In [8]:
import os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

OUT = 'outputs'
os.makedirs(OUT, exist_ok=True)

A = pd.read_csv('outputs/interm_mimic/shadow_economy_estimates.csv').set_index('year_quarter') # mimic
B = pd.read_csv('outputs/hybrid_estimates.csv').set_index('year_quarter') #cda+mimic
C = pd.read_csv('outputs/shadow_economy_results.csv').set_index('year_quarter') # cda
D = pd.read_csv('outputs/shadow_economy_electricity.csv').set_index('quarter') #electricity

SE_A = A['shadow_pct_gdp'].rename('A_classical_MIMIC')
SE_B = B['SE_hybrid'].rename('B_hybrid')
SE_C = C['shadow_method2'].rename('C_standalone_CDA')
SE_D = D['B1_2017_18_38.4pct_total'].rename('D_electricity')
eta_A = A['eta_z_change'].rename('eta_A')

common = SE_A.index.intersection(SE_B.index).intersection(
    SE_C.index).intersection(SE_D.index)
S = pd.concat([SE_A.loc[common], SE_B.loc[common],
               SE_C.loc[common], SE_D.loc[common]], axis=1)


print("=" * 76)
print("PART 1. Cross-method convergence diagnostic")
print("=" * 76)

Z = (S - S.mean()) / S.std(ddof=1)

corr_lvl = Z.corr().round(3)
corr_ma  = Z.rolling(4, min_periods=1).mean().corr().round(3)

print("\nPairwise correlation, standardised levels:")
print(corr_lvl.to_string())
print("\nPairwise correlation, standardised 4-quarter moving averages:")
print(corr_ma.to_string())

dist = 1 - corr_lvl
print("\nDistance matrix (1 - corr):")
print(dist.round(2).to_string())

zd = Z.diff()
sign = np.sign(zd)
convergent = (sign.iloc[:, 0] == sign.iloc[:, 1]) & \
             (sign.iloc[:, 1] == sign.iloc[:, 2]) & \
             (sign.iloc[:, 2] == sign.iloc[:, 3])
print(f"\nQuarters where all 4 series move in the same direction: "
      f"{convergent.sum()} / {len(convergent)-1} "
      f"({100*convergent.sum()/(len(convergent)-1):.0f}%).")

print("\nRMSE between pairs (% of GDP, raw scale):")
for i in range(len(S.columns)):
    for j in range(i+1, len(S.columns)):
        diff = S.iloc[:, i] - S.iloc[:, j]
        rmse = np.sqrt((diff**2).mean())
        print(f"  {S.columns[i]:25s} vs {S.columns[j]:25s}  RMSE = {rmse:6.2f} pp")

corr_lvl.to_csv(f'{OUT}/val_corr_levels.csv')
corr_ma.to_csv(f'{OUT}/val_corr_4qMA.csv')


print("\n" + "=" * 76)
print("PART 2. Block-bootstrap 95% confidence bands for SE_A")
print("=" * 76)

def block_bootstrap(series_changes, block=4, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    n = len(series_changes)
    n_blocks = int(np.ceil(n / block))
    starts = np.arange(n - block + 1)
    boot = np.zeros((n_boot, n))
    for b in range(n_boot):
        chosen = rng.choice(starts, size=n_blocks, replace=True)
        sample = np.concatenate([series_changes.values[s:s+block] for s in chosen])[:n]
        boot[b, :] = sample
    return boot

def calibrate_eta_to_anchor(eta_changes, dates, anchor_period, anchor_value,
                             target_std=5.0):
    eta_level = pd.Series(eta_changes, index=dates).cumsum()
    sigma = eta_level.std(ddof=1)
    mask = (eta_level.index >= anchor_period[0]) & (eta_level.index <= anchor_period[1])
    if not mask.any() or sigma == 0:
        return np.full(len(eta_changes), np.nan)
    return ((eta_level - eta_level[mask].mean()) / sigma * target_std + anchor_value).values

eta_changes_A = eta_A.values
dates_A = eta_A.index
boot_A = block_bootstrap(eta_A, block=4, n_boot=2000, seed=42)

boot_se_A = np.array([
    calibrate_eta_to_anchor(boot_A[b], dates_A,
                             ('2017Q1', '2018Q4'), 38.4)
    for b in range(boot_A.shape[0])
])
ci_lo = np.nanpercentile(boot_se_A, 2.5,  axis=0)
ci_hi = np.nanpercentile(boot_se_A, 97.5, axis=0)
ci_med = np.nanpercentile(boot_se_A, 50, axis=0)

ci_A = pd.DataFrame({
    'SE_A_point':  SE_A.loc[dates_A].values,
    'SE_A_boot_med': ci_med,
    'SE_A_boot_lo':  ci_lo,
    'SE_A_boot_hi':  ci_hi,
}, index=dates_A)
ci_A.to_csv(f'{OUT}/val_bootstrap_SE_A.csv')

print(f"\nBootstrap (B=2000, block=4 quarters) on SE_A:")
print(f"  full-period mean of point estimate : {SE_A.loc[dates_A].mean():.2f}%")
print(f"  full-period mean of bootstrap median: {pd.Series(ci_med).mean():.2f}%")
print(f"  average 95% CI half-width          : {((ci_hi-ci_lo)/2).mean():.2f} pp")


print("\n" + "=" * 76)
print("PART 3. CUSUM stability test on the MIMIC latent (replaces Chow)")
print("=" * 76)

e = eta_A.values
e_std = (e - e.mean()) / e.std(ddof=1)
cusum = np.cumsum(e_std) / np.sqrt(len(e_std))

T = len(e_std)
t = np.arange(1, T+1)
upper = 0.948 * (1 + 2 * t / T)
lower = -upper

breach = np.where((cusum > upper) | (cusum < lower))[0]
if len(breach) == 0:
    print("CUSUM stays inside the 5% bands for the entire 2010-2021 sample.")
    print("=> No evidence of structural instability in the MIMIC's latent dynamics.")
else:
    breach_qs = [eta_A.index[i] for i in breach]
    print(f"CUSUM breaches the 5% band at quarters: {breach_qs}")
    print("=> Evidence of structural instability at those points.")

cusum_df = pd.DataFrame({'cusum': cusum, 'lo': lower, 'hi': upper},
                        index=eta_A.index)
cusum_df.to_csv(f'{OUT}/val_cusum.csv')


print("\n" + "=" * 76)
print("PART 4. Random-Forest *counterfactual* projection 2022Q1-2024Q4")
print("=" * 76)

RAW = pd.read_csv('data/gold_data.csv').drop(
    columns=['Unnamed: 0']).set_index('year_quarter')

feat = pd.DataFrame(index=RAW.index)
feat['d_tax_burden']        = RAW['tax_burden'].diff()
feat['d_rule_of_law']       = RAW['rule_of_law'].diff()
feat['d_direct_tax_share']  = RAW['direct_tax_share'].diff()
feat['d_corruption_perceptions']     = RAW['corruption_perceptions'].diff()
feat['d_government_effectiveness']   = RAW['government_effectiveness'].diff()
feat['dlog_gdp_s21']        = np.log(RAW['gdp_s21']).diff()
feat['d_M0_M2']             = (RAW['cash_demand'] / RAW['deflated_M1']).diff()
feat['d_interest']          = RAW['interest_rate_avg_q'].diff()

train_idx = feat.index[(feat.index >= '2010Q2') & (feat.index <= '2021Q4')]
test_idx  = feat.index[feat.index >= '2022Q1']

X_train = feat.loc[train_idx].dropna()
y_train = SE_A.loc[X_train.index]
X_test  = feat.loc[test_idx].dropna()

print(f"Training: {len(X_train)} quarters, {X_train.shape[1]} features.")
print(f"Test (counterfactual): {len(X_test)} quarters from "
      f"{X_test.index.min()} to {X_test.index.max()}.")

from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=500,
                            max_depth=4,
                            min_samples_leaf=4,
                            oob_score=True,
                            random_state=0,
                            n_jobs=-1)
rf.fit(X_train.values, y_train.values)
oob = rf.oob_score_
y_pred_train = rf.predict(X_train.values)
train_r2 = 1 - ((y_train - y_pred_train)**2).sum() / \
              ((y_train - y_train.mean())**2).sum()
print(f"\nRF training R^2  : {train_r2:.3f}  (in-sample, optimistic)")
print(f"RF OOB score     : {oob:.3f}      (still pre-war only)")

all_tree_pred = np.array([t.predict(X_test.values) for t in rf.estimators_])
y_test_med = np.median(all_tree_pred, axis=0)
y_test_lo  = np.percentile(all_tree_pred, 5,  axis=0)
y_test_hi  = np.percentile(all_tree_pred, 95, axis=0)

cf = pd.DataFrame({
    'CF_median': y_test_med,
    'CF_5pct':   y_test_lo,
    'CF_95pct':  y_test_hi,
}, index=X_test.index)
cf.to_csv(f'{OUT}/val_counterfactual_RF.csv')

print(f"\nCounterfactual median 2022-2024 (% of GDP):")
print(cf['CF_median'].round(1).to_string())

imp = pd.DataFrame({'feature': X_train.columns,
                    'importance': rf.feature_importances_}) \
        .sort_values('importance', ascending=False)
print("\nFeature importances (RF):")
print(imp.round(3).to_string(index=False))
imp.to_csv(f'{OUT}/val_RF_importance.csv', index=False)


def quarter_to_date(q):
    y, qn = q.split('Q')
    month = {'1': 2, '2': 5, '3': 8, '4': 11}[qn]
    return pd.Timestamp(year=int(y), month=month, day=15)

d_a  = pd.DatetimeIndex([quarter_to_date(q) for q in SE_A.index])
d_cf = pd.DatetimeIndex([quarter_to_date(q) for q in cf.index])
d_e  = pd.DatetimeIndex([quarter_to_date(q) for q in eta_A.index])


fig, ax = plt.subplots(figsize=(12, 5))

ax.fill_between(d_a, ci_lo, ci_hi, color='#1f4e79', alpha=0.15,
                label='SE_A 95% bootstrap CI')
ax.plot(d_a, SE_A.values, color='#1f4e79', lw=2.4,
        label='SE_A (classical MIMIC, KIIS-anchored)')
ax.fill_between(d_cf, cf['CF_5pct'].values, cf['CF_95pct'].values,
                color='#c0504d', alpha=0.18,
                label='RF counterfactual 5–95 pct (no-war baseline)')
ax.plot(d_cf, cf['CF_median'].values, color='#c0504d', lw=2.4, ls='--',
        label='RF counterfactual median')
ax.axvline(pd.Timestamp(2022, 2, 1), color='black', lw=0.7, ls=':')
ax.text(pd.Timestamp(2022, 3, 1), ax.get_ylim()[1] * 0.97,
        'Feb 2022\nfull-scale invasion',
        fontsize=8, color='black', va='top')
ax.axvspan(pd.Timestamp(2014, 2, 1), pd.Timestamp(2015, 12, 1),
           alpha=0.07, color='red', zorder=0)
ax.set_title('SE_A with bootstrap CI and RF counterfactual extension\n'
             '(2022–2024 are not war-time estimates)',
             fontsize=11, pad=10)
ax.set_ylabel('% of GDP', fontsize=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.legend(loc='upper left', fontsize=8, frameon=True)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUT}/plot1_bootstrap_CI_RF.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"\nSaved: {OUT}/plot1_bootstrap_CI_RF.png")


fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(d_e, cusum, color='#1f4e79', lw=2.0, label='CUSUM(η_t)')
ax.plot(d_e, upper, color='gray', lw=1, ls='--', label='5% bands')
ax.plot(d_e, lower, color='gray', lw=1, ls='--')
ax.axhline(0, color='black', lw=0.4)
ax.set_title('CUSUM stability test on the MIMIC latent\n'
             '(no breaches → pre-war structure stable)',
             fontsize=11, pad=10)
ax.set_ylabel('CUSUM', fontsize=10)
ax.legend(loc='upper right', fontsize=8)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{OUT}/plot2_cusum.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUT}/plot2_cusum.png")


fig, ax = plt.subplots(figsize=(6, 5))

im = ax.imshow(corr_lvl.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
labels = ['A. MIMIC', 'B. Hybrid', 'C. CDA', 'D. Electricity']
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
for i in range(len(corr_lvl.index)):
    for j in range(len(corr_lvl.columns)):
        ax.text(j, i, f'{corr_lvl.values[i,j]:+.2f}',
                ha='center', va='center', fontsize=10,
                color='white' if abs(corr_lvl.values[i,j]) > 0.5 else 'black')
plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04, label='Pearson ρ')
ax.set_title('Cross-method correlation matrix\n(z-scored levels, 2010Q2–2021Q4)',
             fontsize=11, pad=10)

plt.tight_layout()
plt.savefig(f'{OUT}/plot3_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"Saved: {OUT}/plot3_correlation_matrix.png")

PART 1. Cross-method convergence diagnostic

Pairwise correlation, standardised levels:
                   A_classical_MIMIC  B_hybrid  C_standalone_CDA  D_electricity
A_classical_MIMIC              1.000     0.356            -0.306         -0.040
B_hybrid                       0.356     1.000             0.002          0.041
C_standalone_CDA              -0.306     0.002             1.000          0.422
D_electricity                 -0.040     0.041             0.422          1.000

Pairwise correlation, standardised 4-quarter moving averages:
                   A_classical_MIMIC  B_hybrid  C_standalone_CDA  D_electricity
A_classical_MIMIC              1.000     0.451            -0.537          0.009
B_hybrid                       0.451     1.000            -0.530         -0.122
C_standalone_CDA              -0.537    -0.530             1.000          0.431
D_electricity                  0.009    -0.122             0.431          1.000

Distance matrix (1 - corr):
                   A